# Turner angle A/B — current formula (store) vs closed-set form

**Current (store channel):**

    Tu = arctan[ ρ₀(β²|∇S|² − α²|∇T|²) / (−gradrho2/ρ₀) ]

Numerator: linear-EOS model from T,S squares.  Denominator: the
MEASURED JMD95 `gradrho2`.  Two measurement routes — at compensated
fronts both are tiny residuals of large cancellations, their errors
decorrelate, and the ratio's sign is a coin flip → ±90° speckle.

**Closed-set (rows 3–4):** measure the cross term directly and build
BOTH parts from the same staggered T,S samples:

    A = α²|∇T|²,  B = β²|∇S|²,  X = αβ(∇T·∇S)
    Tu = arctan[ ρ₀(B − A) / (−ρ₀(A + B − 2X)) ]

`X` uses co-located native products (Tx·Sx on u-points, Ty·Sy on
v-points; interp AFTER multiplying), so the identity
(βS′−αT′)² = B′−2X′+A′ holds exactly per sample and survives the
(linear) interpolation.  The denominator is a mean of true
non-negative squares — no fake zeros, no sign noise.  What remains
undefined is REAL: perfectly compensated water (N→0, D→0) — masked
below a visible floor.  `gradrho2` the channel is untouched (it
keeps the full-EOS ρ differencing; the difference to the linear
denominator is the cabbeling content).

Limits preserved: pure T front → +45°; pure S front → −45°;
near-compensation → ±90°; exact compensation → NaN (mask).

## Section 1 — LOAD the store channel + grid

Rows 1–2 come from the SAVED frontal_structure store (regenerate it
first if stale — see the frontal_structure notebook Section 1).

In [ ]:
# Section 1: frontal_structure store reader + shared grid.
import numpy as np
import matplotlib.pyplot as plt
import cmocean.cm as cmo

import dbof.io.filesystems as filesystems
import dbof.global_dataset_creation.zarr_dataset_global as zarr_dataset
import dbof.global_dataset_creation.zarr_grid_global as zarr_grid
from dbof.global_dataset_creation.subset_definitions import (
    get_subset_definition,
)

S3_ENDPOINT = "https://s3-west.nrp-nautilus.io"
BUCKET      = "dbof"
FOLDER      = "surface_fields"
RUN_ID      = "field_validation_v1"
PIPELINE    = "SURF"
DATE        = "2012-11-09 12:00:00"
DATE_PREFIX = "20121109_120000"

defn = get_subset_definition(PIPELINE, "frontal_structure")
fs, _ = filesystems.create_s3_filesystems(S3_ENDPOINT)
reader = zarr_dataset.GlobalZarrDatasetReader(
    bucket=BUCKET, folder=FOLDER, run_id=RUN_ID,
    dataset_name=defn["dataset_name"], date_prefix=DATE_PREFIX,
    fs=fs,
)
fs_grid, _ = filesystems.create_s3_filesystems(S3_ENDPOINT)
grid_reader = zarr_grid.GlobalGridZarrReader(
    bucket=BUCKET, folder="LLC4320_GRID_2D",
    dataset_name="llc4320_grid.zarr", fs=fs_grid,
)
XC, YC = grid_reader.lon, grid_reader.lat
print(f"store channels: {reader.channel_names}")

## Section 2 — Region selection

In [ ]:
# Section 2: pick the region; re-run from here after changing it.
from dbof.plotting import regions

REGION       = "gulf_stream"   # <-- change me
ZOOM_HALF_KM = 100.0           # 200x200 km zoom box

_zoomable = [n for n, r in regions.REGIONS.items() if "zoom" in r]
assert REGION in _zoomable, f"pick one of {_zoomable}"
print(f"region: {REGION}  (options: {_zoomable})")

## Section 3 — Closed-set Turner angle, live

The candidate `calculate_grad_dot_tracer` is defined INLINE here
(notebook-only until the A/B is accepted); it is the verbatim
candidate for `utils/native_gradient.py`.

In [ ]:
# Section 3a: live closed-set Tu (lazy).
import dbof.preprocessing.calculate_fields as calculate_fields
import dbof.utils.native_gradient as ng
from dbof.cli.generate_global import load_snapshot
from dbof.global_dataset_creation.data_sources import get_data_source
from dbof.global_dataset_creation.grid_setup import set_up_grid
from dbof.preprocessing.physical_constants import (
    ALPHA, BETA, RHO0_REFERENCE,
)

ds_grid, land_mask, xgrid = set_up_grid(PIPELINE, None)
ds_raw, ds_merge, it = load_snapshot(
    PIPELINE, DATE, ds_grid, ["Theta", "Salt"],
    surface_only=False, data_source=get_data_source(PIPELINE),
)
print(f"OSN iteration {it}  (store iteration {reader.iteration})")


def calculate_grad_dot_tracer(da_a, da_b, ds_grid, grid):
    """∇a·∇b with the products formed ON the staggered points.

    ax and bx are CO-LOCATED on the u-points (ay, by on the
    v-points), so the products need no pre-interpolation — the
    same-order-of-operations principle as
    ``calculate_grad_squared_tracer`` (∇a·∇a is that function).
    Verbatim candidate for utils/native_gradient.py.
    Inputs: da_a, da_b (tracer-point DataArrays); ds_grid (dxC,
    dyC); grid (xgcm).  Outputs: ∇a·∇b at tracer points, lazy.
    Generated by LH and Claude
    """
    ax = grid.diff(da_a, 'X') / ds_grid.dxC
    bx = grid.diff(da_b, 'X') / ds_grid.dxC
    ay = grid.diff(da_a, 'Y') / ds_grid.dyC
    by = grid.diff(da_b, 'Y') / ds_grid.dyC
    return (grid.interp(ax * bx, 'X', boundary='fill')
            + grid.interp(ay * by, 'Y', boundary='fill'))


T, S = ds_merge.Theta, ds_merge.Salt
A = ALPHA**2 * ng.calculate_grad_squared_tracer(T, ds_merge, xgrid)
B = BETA**2 * ng.calculate_grad_squared_tracer(S, ds_merge, xgrid)
X = ALPHA * BETA * calculate_grad_dot_tracer(T, S, ds_merge, xgrid)

N = RHO0_REFERENCE * (B - A)
D_lin = -RHO0_REFERENCE * (A + B - 2.0 * X)   # ≤ 0 by construction
Tu_new = np.degrees(np.arctan(N / D_lin))     # mask applied later

live_map = {"turner_new": Tu_new, "D_lin": D_lin}
print("lazy closed-set Tu ready")

In [ ]:
# Section 3b: stitch + slice (shared plumbing) and store slice.
from dbof.plotting.live_fields import stitch_and_slice, slice_store

region_arrays = stitch_and_slice(
    live_map, ds_raw, ds_merge, XC, YC, [REGION], batch=2)
region_arrays.update(
    slice_store(reader, ["turner_angle"], XC, YC, [REGION]))
print("sliced:", sorted(region_arrays))

## Section 4 — Mask floor

The closed-set Tu is undefined where the water is exactly
compensated (D_lin → 0).  `FLOOR_PCT` masks the weakest x % of
|D_lin| in the region — tune and re-run from here.

In [ ]:
# Section 4: mask the undefined core (regional, tunable).
FLOOR_PCT = 1.0    # mask the weakest x% of |D_lin| — tune me

_x, _y, _d = region_arrays["D_lin"][REGION]
_xn, _yn, _tu = region_arrays["turner_new"][REGION]
_absd = np.abs(_d)
_floor = np.nanpercentile(_absd[np.isfinite(_absd)], FLOOR_PCT)
tu_masked = np.where(_absd > _floor, _tu, np.nan)
_frac = 100.0 * np.mean(~np.isfinite(tu_masked)
                        & np.isfinite(_tu))
print(f"floor = {_floor:.3e} (P{FLOOR_PCT}); "
      f"masked {_frac:.2f}% of ocean pixels")
region_arrays["turner_new_masked"] = {
    REGION: (_xn, _yn, tu_masked)}

## Section 5 — A/B figure

Rows 1–2: STORE Tu (current formula), full + 200×200 km zoom.
Rows 3–4: closed-set Tu (masked), full + zoom.  Fixed ±90° balance
scale (zero at white) on every panel.  Look for the red/blue
salt-and-pepper in the store rows disappearing in rows 3–4; gray
pixels in rows 3–4 are the masked undefined core.

In [ ]:
# Section 5: 4-row comparison (store vs closed-set).
from dbof.plotting.pipeline_grids import LAND_COLOR

ROWS = [
    ("store (full)", "turner_angle", False),
    ("store (zoom)", "turner_angle", True),
    ("closed-set (full)", "turner_new_masked", False),
    ("closed-set (zoom)", "turner_new_masked", True),
]

fig, axes = plt.subplots(4, 1, figsize=(9.5, 4 * 3.4))
pm = None
for ax, (label, fld, zoom) in zip(axes, ROWS):
    x, y, arr = region_arrays[fld][REGION]
    if zoom:
        x, y, arr = regions.crop_zoom(x, y, arr, REGION,
                                      half_km=ZOOM_HALF_KM)
    ax.set_facecolor(LAND_COLOR)
    pm = ax.pcolormesh(x, y, arr, cmap=cmo.balance,
                       vmin=-90, vmax=90, shading="nearest")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_ylabel(label, fontsize=10)
fig.colorbar(pm, ax=list(axes), orientation="horizontal",
             shrink=0.8, pad=0.02, label="Turner angle (deg)")
fig.suptitle(f"turner_angle \u2014 store (rows 1-2) vs "
             f"closed-set (rows 3-4), {REGION}", fontsize=13)
plt.show()

## Section 6 — Distributions

The speckle signature in the PDF: spurious ±90° pileup.  Expect the
closed-set curve to keep the physical ±45° structure and shrink the
±90° spikes to the (masked-adjacent) genuinely compensated water.

In [ ]:
# Section 6: Tu histograms, store vs closed-set (same region).
_, _, tu_store = region_arrays["turner_angle"][REGION]
bins = np.linspace(-90, 90, 181)
fig, ax = plt.subplots(figsize=(9, 4))
for arr, label, colr in [
    (tu_store, "store (current formula)", "firebrick"),
    (tu_masked, "closed-set (masked)", "steelblue"),
]:
    v = arr[np.isfinite(arr)]
    ax.hist(v, bins=bins, histtype="step", density=True,
            label=f"{label}  (n={v.size})", color=colr)
ax.set_xlabel("Turner angle (deg)")
ax.set_ylabel("density")
ax.axvline(45, color="gray", lw=0.5)
ax.axvline(-45, color="gray", lw=0.5)
ax.legend()
ax.set_title(f"Tu distributions, {REGION}")
plt.show()

## Findings

*(fill in after running)*

- Store ±90° speckle vs closed-set:
- Masked fraction at the chosen floor:
- PDF ±90° pileup change:
- Decision (adopt closed-set + mask in `calculate_fields`?):